# 08. thinking（考えてから答える）のオン・オフを比べる

このノートは、[colab-oss-lab](https://github.com/moruku36/colab-oss-lab) の実験 08 です。

02 で L4 に載せた `google/gemma-4-31B-it`（4bit）を使い、**同じ問題を thinking オフ / オン で解かせて**比べます。

- thinking オフ: すぐ答える
- thinking オン: 答える前に、頭の中のメモ（思考）を書いてから答える

測るもの:

1. 正答数（答えが決まっている問題 5 問）
2. 1問あたりの時間と、出したトークン数
3. VRAM のピーク

「想定どおり」とは:

- thinking オンの正答数が、オフ以上になる
- thinking オンは時間がかかる（トークンが増える）
- どちらも L4 の VRAM（22GB）に収まる

所要時間の目安: 読み込み 約6分 + 問題 10〜30分。

---

## 実行する前に

1. **ランタイム → ランタイムのタイプを変更 → L4 GPU → Save**
2. 上から順に ▶（または「すべてのセルを実行」）
3. 終わったら **ランタイム → セッションを管理 → 解放**

## 1. 準備（02 と同じ）

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
import re, time, torch, transformers, bitsandbytes
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "ランタイムを L4 GPU にしてください"
gpu_name = torch.cuda.get_device_name(0)
vram_total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name, round(vram_total_gb, 1), "GB")
print("transformers:", transformers.__version__, "/ bitsandbytes:", bitsandbytes.__version__)

In [ ]:
MODEL_ID = "google/gemma-4-31B-it"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, dtype=torch.bfloat16, device_map={"": 0},
)
load_min = (time.time() - t0) / 60
print("読み込み完了:", round(load_min, 1), "分 / VRAM", round(torch.cuda.memory_allocated() / 1024**3, 1), "GB")

## 2. 問題

答えが1つに決まる問題を5つ用意しました。最後に「答え: ○○」と書かせて、数字を取り出して採点します。

In [ ]:
QUESTIONS = [
    ("ある数に3を足して2倍すると、その数の3倍より4小さくなります。ある数はいくつですか。", 10),
    ("1から100までの整数のうち、3でも5でも割り切れないものはいくつありますか。", 53),
    ("英単語 strawberry の中に、アルファベットの r は何個含まれていますか。", 3),
    ("A、B、C、D の4人が横一列に並びます。AとBが隣り合わない並び方は何通りですか。", 12),
    ("時計が3時15分を指しているとき、長針と短針がつくる小さいほうの角は何度ですか。", 7.5),
]
SYSTEM = "You are a helpful assistant. Answer in Japanese. 最後の行に必ず「答え: <数字>」の形で答えだけを書いてください。"
MAX_NEW = {False: 512, True: 2048}

## 3. 解かせる関数と採点

In [ ]:
SPECIAL = re.compile(r"<\|?[a-zA-Z_]+\|?>")

def split_output(raw):
    # Gemma 4 は <|channel>thought\n(思考)<channel|>(答え) の形で出す
    if "<channel|>" in raw:
        thought, final = raw.split("<channel|>", 1)
        thought = thought.split("<|channel>thought", 1)[-1]
        truncated = False
    else:
        thought, final, truncated = raw.split("<|channel>thought", 1)[-1], "", True  # 思考の途中で打ち切られた
    return SPECIAL.sub("", thought).strip(), SPECIAL.sub("", final).strip(), truncated

def extract_answer(final):
    m = re.findall(r"答え\s*[:：]\s*\**\s*([0-9]+(?:\.[0-9]+)?)", final)
    if not m:
        m = re.findall(r"([0-9]+(?:\.[0-9]+)?)", final)
    return float(m[-1]) if m else None

def solve(question, thinking):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=thinking,
    ).to(model.device)
    n_in = inputs["input_ids"].shape[-1]
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=MAX_NEW[thinking], do_sample=False)
    torch.cuda.synchronize(); sec = time.time() - t0
    gen = out[0][n_in:]
    raw = processor.decode(gen, skip_special_tokens=False)
    thought, final, truncated = split_output(raw)
    n_thought = len(processor.tokenizer(thought)["input_ids"]) if thought else 0
    return dict(thought=thought, final=final, truncated=truncated,
                n_out=int(gen.shape[-1]), n_thought=n_thought, sec=sec)

## 4. オフ → オン の順に解かせる

In [ ]:
torch.cuda.reset_peak_memory_stats()
results = []
for i, (q, ans) in enumerate(QUESTIONS, 1):
    for thinking in (False, True):
        r = solve(q, thinking)
        got = extract_answer(r["final"])
        r.update(q=q, expected=ans, got=got, thinking=thinking,
                 correct=(got is not None and abs(got - ans) < 1e-6))
        results.append(r)
        tag = "オン" if thinking else "オフ"
        mark = "○" if r["correct"] else "×"
        print(f"Q{i} thinking {tag}: {mark} 答え={got}（正解 {ans}）"
              f" / {r['n_out']} トークン（うち思考 {r['n_thought']}） / {r['sec']:.1f} 秒"
              + (" / 打ち切り" if r["truncated"] else ""))
vram_peak_gb = torch.cuda.max_memory_allocated() / 1024**3
print("VRAM ピーク:", round(vram_peak_gb, 1), "GB")

## 5. 集計して、実行記録を出す

In [ ]:
from datetime import datetime, timezone, timedelta

def summary(thinking):
    rs = [r for r in results if r["thinking"] == thinking]
    return dict(
        correct=sum(r["correct"] for r in rs), n=len(rs),
        sec=sum(r["sec"] for r in rs) / len(rs),
        tok=sum(r["n_out"] for r in rs) / len(rs),
        tps=sum(r["n_out"] for r in rs) / sum(r["sec"] for r in rs),
        truncated=sum(r["truncated"] for r in rs),
    )
off, on = summary(False), summary(True)
checks = {
    "thinking オンの正答数がオフ以上": on["correct"] >= off["correct"],
    "thinking オンのほうが時間がかかる": on["sec"] > off["sec"],
    "VRAM が 22GB に収まった": vram_peak_gb < vram_total_gb,
}
ok = all(checks.values())

now = datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M JST")
L = [
    "# 実行記録: 08 thinking のオン・オフ",
    "",
    f"- 実行日: {now}",
    "- 実行場所: Google Colab",
    f"- GPU: {gpu_name} / VRAM {round(vram_total_gb, 1)} GB",
    f"- モデル: {MODEL_ID}（bitsandbytes 4bit NF4）",
    f"- transformers {transformers.__version__} / bitsandbytes {bitsandbytes.__version__} / torch {torch.__version__}",
    "- 生成設定: greedy（do_sample=False）、max_new_tokens オフ 512 / オン 2048",
    f"- 読み込み時間: {round(load_min, 1)} 分",
    f"- VRAM ピーク: {round(vram_peak_gb, 1)} GB",
    f"- 想定どおりか: {'はい' if ok else 'いいえ'}",
    "",
    "## まとめ",
    "",
    "| | thinking オフ | thinking オン |",
    "|---|---|---|",
    f"| 正答数 | {off['correct']} / {off['n']} | {on['correct']} / {on['n']} |",
    f"| 1問あたりの時間 | {off['sec']:.1f} 秒 | {on['sec']:.1f} 秒 |",
    f"| 1問あたりの出力トークン | {off['tok']:.0f} | {on['tok']:.0f} |",
    f"| 速度 | {off['tps']:.1f} トークン/秒 | {on['tps']:.1f} トークン/秒 |",
    f"| 思考の途中で打ち切り | {off['truncated']} | {on['truncated']} |",
    "",
    "## 判定",
    "",
] + [f"- [{'x' if v else ' '}] {k}" for k, v in checks.items()] + [
    "",
    "## 問題ごと",
    "",
    "| # | 問題 | 正解 | オフ | オン | オン思考トークン | オフ秒 | オン秒 |",
    "|---|---|---|---|---|---|---|---|",
]
for i, (q, ans) in enumerate(QUESTIONS, 1):
    a = next(r for r in results if r["q"] == q and not r["thinking"])
    b = next(r for r in results if r["q"] == q and r["thinking"])
    fmt = lambda r: f"{'○' if r['correct'] else '×'} {r['got']}"
    L.append(f"| {i} | {q} | {ans} | {fmt(a)} | {fmt(b)} | {b['n_thought']} | {a['sec']:.1f} | {b['sec']:.1f} |")
L += ["", "## 返事（最終回答の部分）", ""]
for i, (q, ans) in enumerate(QUESTIONS, 1):
    L += [f"### Q{i}. {q}", ""]
    for r in results:
        if r["q"] != q:
            continue
        L += [f"thinking {'オン' if r['thinking'] else 'オフ'}:", "", "```", r["final"] or "（最終回答なし）", "```", ""]
report = "\n".join(L)
print(report)

## おまけ: 思考の中身をのぞく

thinking オンのとき、モデルが答える前に何を書いていたかを表示します（長いので先頭 1500 文字だけ）。

In [ ]:
for i, r in enumerate([r for r in results if r["thinking"]], 1):
    print(f"===== Q{i} の思考（{r['n_thought']} トークン） =====")
    print(r["thought"][:1500])
    print()

## 終わったら

**ランタイム → セッションを管理 → 解放** を必ず押してください。